In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f = xW + b $$
$$ \frac{\partial f}{\partial x} = W $$
$$ \frac{\partial f}{\partial W} = x $$
$$ \frac{\partial f}{\partial b} = 1 $$

In [ ]:
import torch.autograd as autograd
import torch.nn as nn
import torch as tr

import import_ipynb
from common import assert_eq # type: ignore


def linear(x: tr.Tensor, W: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
    # (samples, output) = (samples, features) @ (features, output) + (output,)
    return tr.addmm(b, x, W)


class LinearFunction(autograd.Function):
    @staticmethod
    def forward(ctx, x: tr.Tensor, W: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(x, W)
        return linear(x, W, b)

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor, tr.Tensor]:
        x, W = ctx.saved_tensors
        samples, features = x.shape
        output = W.shape[1]
       
        # (samples, features) = (samples, output) @ (output, features)
        dF_dx = dF_df @ W.T
        assert_eq(dF_dx.shape, x.shape)
        assert_eq(dF_dx.shape, (samples, features))
        
        # (features, output) = (features, samples) @ (samples, output)
        dF_dW = x.t() @ dF_df          
        assert_eq(dF_dW.shape, W.shape)
        assert_eq(dF_dW.shape, (features, output))
        
        # (output,) * (output,) -> (output, )
        dF_db = dF_df.sum(dim=0)     
        assert_eq(dF_db.shape, (output,))
        
        return (dF_dx, dF_dW, dF_db)
    

# `LinearFunction` implements the actual operator.
# `Linear(Module)` is just a wrapper for nicer API.
#
# For consistency with the nn.Linear, weights is transposed, y = x @ W.t() + b
#
class Linear(nn.Module):
    def __init__(self, in_features, out_features, W=None, b=None):
        super().__init__()

        if W is not None:
            assert_eq(W.shape, (out_features, in_features))
            self.weight = nn.Parameter(W)
        else:
            self.weight = nn.Parameter(tr.randn(out_features, in_features) * 0.01)
        
        if b is not None:
            assert_eq(b.shape, (out_features,))
            self.bias = nn.Parameter(b)
        else:
            self.bias = nn.Parameter(tr.zeros(out_features))

    def forward(self, x):
        return LinearFunction.apply(x, self.weight.T, self.bias)
    

def test_linear_1() -> None:
    tr.manual_seed(0)

    samples = 10
    in_features = 3
    out_features = 4
    x = tr.randn(samples, in_features, dtype=tr.float32, requires_grad=True)
    W = tr.randn(in_features, out_features, dtype=tr.float32, requires_grad=True)
    b = tr.randn(out_features, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    W1 = W.clone().detach().requires_grad_(True)
    b1 = b.clone().detach().requires_grad_(True)
    F1 = linear(x1, W1, b1).sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    W2 = W.clone().detach().requires_grad_(True)
    b2 = b.clone().detach().requires_grad_(True)
    F2 = (x2 @ W2 + b2).sum()
    F2.backward()

    assert_eq(x1, x2, atol=0.001)
    assert_eq(x1.grad, x2.grad, atol=0.001)
    assert_eq(W1, W2, atol=0.001)
    assert_eq(W1.grad, W2.grad, atol=0.001)
    assert_eq(b1, b2, atol=0.001)
    assert_eq(b1.grad, b2.grad, atol=0.001)


def test_linear_2() -> None:
    tr.manual_seed(0)

    samples = 3
    in_features = 4
    out_features = 5
    x = tr.randn(samples, in_features, dtype=tr.float32, requires_grad=True)
    W = tr.randn(out_features, in_features, dtype=tr.float32, requires_grad=True)
    b = tr.randn(out_features, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    W1 = W.clone().detach().requires_grad_(True)
    b1 = b.clone().detach().requires_grad_(True)
    linear = Linear(in_features, out_features, W=W1, b=b1)
    F1 = linear(x1).sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    W2 = W.clone().detach().requires_grad_(True)
    b2 = b.clone().detach().requires_grad_(True)
    nn_linear = nn.Linear(in_features, out_features)

    with tr.no_grad():
        nn_linear.weight.copy_(W1)
        nn_linear.bias.copy_(b1)

    F2 = nn_linear(x).sum()
    F2.backward()

    assert_eq(x1, x, atol=0.001)
    assert_eq(x1.grad, x.grad, atol=0.001)
    assert_eq(linear.weight.data, nn_linear.weight.data, atol=0.001)
    assert_eq(linear.weight.grad, nn_linear.weight.grad, atol=0.001)
    assert_eq(linear.bias.data, nn_linear.bias.data, atol=0.001)
    assert_eq(linear.bias.grad, nn_linear.bias.grad, atol=0.001)


if __name__ == "__main__":
    test_linear_1()
    test_linear_2()
